# Basics 1: reading and preparing data

crispyx works on `.h5ad` files on disk: every function reads the matrix in blocks and writes its result to a new file, so memory stays small however large the screen is. This notebook is the first of three that together run or name every public function:

1. **Reading and preparing data** (this notebook): get counts into an `.h5ad` file, attach labels, check and tidy it.
2. [QC, normalisation and embedding](basics_2_qc_embedding.ipynb): decide what to keep and look at the cells.
3. [Effects and differential expression](basics_3_effects_tests.ipynb): estimate and test perturbation effects.

Each notebook runs on its own, in under a minute, on the dataset bundled with the repository (`data/Adamson_subset.h5ad`: 1,716 cells, 11,630 genes, two targets and a control). The table at the end lists every function of this area and where it is shown.

## Setup

Install crispyx (`pip install crispyx`, or `pip install -e .[test]` from a clone). The cell below imports it and sets the paths.

In [1]:
import os

# Progress bars off, so the saved notebook keeps only crispyx's summary lines; tqdm
# reads this when first imported, so it is set before anything imports tqdm.
os.environ.setdefault('TQDM_DISABLE', '1')

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse as sp

# Paths stay relative to this notebook's directory so that the paths crispyx
# echoes in its output are the same on every machine.
ROOT = Path('../..')
sys.path.insert(0, str(ROOT / 'src'))
DATA_PATH = ROOT / 'data' / 'Adamson_subset.h5ad'
OUTPUT_DIR = Path('tutorial_outputs') / 'basics_data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings('ignore', message='IProgress not found')

import crispyx as cx
print(f'crispyx {cx.__version__}')

crispyx 0.2.11


## 1. Open an h5ad file

`cx.read_h5ad_ondisk` prints a preview and returns a `cx.AnnData`: a read-only handle whose matrix stays on disk. Its `obs` and `var` show their first rows with `head()` and load in full with `load()`.

In [2]:
adata = cx.read_h5ad_ondisk(DATA_PATH)

AnnData object with n_obs × n_vars = 1716 × 11630 backed at '../../data/Adamson_subset.h5ad'
    obs: 'perturbation', 'n_genes', 'dataset', 'celltype'
    var: 'Ensembl_ID'
First obs rows:
                 perturbation  n_genes  dataset celltype
Cell_barcodes                                           
AAACATACAAGATG-1      control     2914  Adamson     K562
AAACATTGCATTGG-1      control     3999  Adamson     K562
AAACGGCTAATGCC-1       AMIGO3     3947  Adamson     K562
AAAGACGAACTCAG-1      control     2713  Adamson     K562
AAAGAGACTCGCCT-1      control     3678  Adamson     K562
First var rows:
                  Ensembl_ID
Gene_symbol                 
LINC00115    ENSG00000225880
NOC2L        ENSG00000188976
KLHL17       ENSG00000187961
HES4         ENSG00000188290
ISG15        ENSG00000187608


In [3]:
adata.obs.load()['perturbation'].value_counts()

perturbation
AMIGO3     750
control    500
AARS       466
Name: count, dtype: int64

`cx.load_obs` and `cx.load_var` read one table without opening the matrix, and `cx.write_obs` / `cx.write_var` (section 4) write an edited table back. When another library needs anndata's own backed object, `cx.read_backed` opens one with every sparse layer left on disk.

In [4]:
obs, var = cx.load_obs(DATA_PATH), cx.load_var(DATA_PATH)
print('obs:', obs.shape, list(obs.columns))
print('var:', var.shape, list(var.columns))

backed = cx.read_backed(DATA_PATH)
print(type(backed).__name__, 'backed:', backed.isbacked, '| X on disk as', type(backed.X).__name__)

obs: (1716, 4) ['perturbation', 'n_genes', 'dataset', 'celltype']
var: (11630, 1) ['Ensembl_ID']
AnnData backed: True | X on disk as _CSRDataset


## 2. From a MatrixMarket folder

Most pipelines deliver a folder with `matrix.mtx`, `barcodes.tsv` and `features.tsv`. To have one to read, the next cell exports the bundled counts in that layout and adds what a CRISPR run has beside the genes: guide-capture rows. **The guide counts are simulated** from the dataset's labels (two guides per target, two non-targeting guides, a background, 5% of cells with a second guide and 3% with none); everything else is the real data.

In [5]:
rng = np.random.default_rng(0)
counts = backed.X[:]            # 1,716 cells: small enough to hold for this export
backed.file.close()

guide_names = ['AARS_g1', 'AARS_g2', 'AMIGO3_g1', 'AMIGO3_g2', 'NTC_g1', 'NTC_g2']
n_cells = len(obs)
target = obs['perturbation'].astype(str).replace({'control': 'NTC'}).to_numpy()
own = np.array([guide_names.index(f'{t}_g{g}') for t, g in zip(target, rng.integers(1, 3, n_cells))])
guide_counts = rng.poisson(0.2, (n_cells, len(guide_names)))                 # background
guide_counts[np.arange(n_cells), own] += rng.poisson(40, n_cells)            # the cell's own guide
second = np.flatnonzero(rng.random(n_cells) < 0.05)                          # a second guide: a multiplet
guide_counts[second, rng.integers(0, len(guide_names), second.size)] += rng.poisson(40, second.size)
empty = rng.random(n_cells) < 0.03                                           # no guide captured
guide_counts[empty] = rng.poisson(0.2, (empty.sum(), len(guide_names)))

mex = OUTPUT_DIR / 'mex'
mex.mkdir(exist_ok=True)
scipy.io.mmwrite(mex / 'matrix.mtx', sp.vstack([counts.T, sp.csr_matrix(guide_counts.T)]).astype(int))  # features x barcodes
pd.DataFrame({
    'id': list(var['Ensembl_ID']) + guide_names,
    'name': list(var.index) + guide_names,
    'type': ['Gene Expression'] * len(var) + ['CRISPR Guide Capture'] * len(guide_names),
}).to_csv(mex / 'features.tsv', sep='\t', header=False, index=False)
pd.Series(obs.index).to_csv(mex / 'barcodes.tsv', header=False, index=False)
print(sorted(p.name for p in mex.iterdir()))

['barcodes.tsv', 'features.tsv', 'matrix.mtx']


`cx.read_10x_mtx` takes the names and defaults of Scanpy's `sc.read_10x_mtx` and streams the folder into an `.h5ad` file instead of reading the matrix into memory. Here `gex_only=False` keeps the guide rows, which Scanpy's default (and crispyx's) drops.

In [6]:
screen = cx.read_10x_mtx(mex, output_path=OUTPUT_DIR / 'screen_all_features.h5ad', gex_only=False)
feature_types = cx.load_var(screen.path)['feature_types']
feature_types.value_counts()

[cx] read_10x_mtx: tutorial_outputs/basics_data/mex/matrix.mtx -> tutorial_outputs/basics_data/screen_all_features.h5ad: 1,716 cells x 11,636 genes


feature_types
Gene Expression         11630
CRISPR Guide Capture        6
Name: count, dtype: int64

The two kinds of rows are different measurements, so they go to separate files. `cx.write_filtered_subset` streams any subset of cells and genes to a new file.

In [7]:
counts_path, guides_path = OUTPUT_DIR / 'screen_counts.h5ad', OUTPUT_DIR / 'screen_guides.h5ad'
every_cell = np.ones(screen.shape[0], dtype=bool)
for path, kind in ((counts_path, 'Gene Expression'), (guides_path, 'CRISPR Guide Capture')):
    cx.write_filtered_subset(screen.path, cell_mask=every_cell, gene_mask=(feature_types == kind).to_numpy(), output_path=path)
    print(path, cx.AnnData(path).shape)

tutorial_outputs/basics_data/screen_counts.h5ad (1716, 11630)
tutorial_outputs/basics_data/screen_guides.h5ad (1716, 6)


`cx.read_mtx` reads a matrix file on its own (cells and genes are then numbered); `transpose=True` says its rows are genes.

In [8]:
bare = cx.read_mtx(mex / 'matrix.mtx', output_path=OUTPUT_DIR / 'bare_matrix.h5ad', transpose=True)
print(bare.shape, list(cx.load_obs(bare.path).index[:3]))

[cx] read_mtx: tutorial_outputs/basics_data/mex/matrix.mtx -> tutorial_outputs/basics_data/bare_matrix.h5ad: 1,716 cells x 11,636 genes
(1716, 11636) ['0', '1', '2']


## 3. From a text table

Published screens often ship counts as a CSV with genes as rows. `cx.read_csv` (and `cx.read_text` for any other delimiter) follow Scanpy's `sc.read_csv` / `sc.read_text` and stream the table; `transpose=True` replaces Scanpy's `.T`. The [targeted-screen tutorial](residual_effects_targeted_feng.ipynb) converts a 4.5 GB table this way.

In [9]:
table = pd.DataFrame(counts[:200, :300].T.toarray().astype(int), index=var.index[:300], columns=obs.index[:200])
table.to_csv(OUTPUT_DIR / 'counts_table.csv.gz')            # 300 genes x 200 cells
from_csv = cx.read_csv(OUTPUT_DIR / 'counts_table.csv.gz', output_path=OUTPUT_DIR / 'from_csv.h5ad', transpose=True)
print(from_csv.shape, 'equal to the source:', (from_csv.to_memory().X != counts[:200, :300]).nnz == 0)

[cx] read_csv: tutorial_outputs/basics_data/counts_table.csv.gz -> tutorial_outputs/basics_data/from_csv.h5ad: 200 cells x 300 genes
(200, 300) equal to the source: True


## 4. Assign guides and attach labels

`cx.pp.assign_guides` gives each cell the guide that dominates its guide-capture counts (at least `min_umi` UMIs and `min_ratio` times the runner-up), and `NA` where none does. An unassigned cell is not a control: whether such cells may join the controls is a question for the data, answered in the [control-checks tutorial](control_checks.ipynb).

In [10]:
guide = cx.pp.assign_guides(guides_path)
guide.value_counts(dropna=False)

[cx] pp.assign_guides: Done  1608/1716 cells assigned a guide (top guide >= 3 UMIs and >= 2x the runner-up); 108 unassigned


guide
AMIGO3_g2    363
AMIGO3_g1    345
NTC_g2       247
AARS_g1      227
NTC_g1       215
AARS_g2      211
<NA>         108
Name: count, dtype: int64

Edit `obs` as a DataFrame and write it back with `cx.write_obs`; the matrix is not read. The unassigned cells are then left out of the file the analysis uses.

In [11]:
obs_new = cx.load_obs(counts_path)
obs_new['guide'] = guide.astype('category')
cx.write_obs(counts_path, obs_new)

assigned_path = OUTPUT_DIR / 'screen_assigned.h5ad'
cx.write_filtered_subset(counts_path, cell_mask=guide.notna().to_numpy(),
                         gene_mask=np.ones(len(cx.load_var(counts_path)), dtype=bool), output_path=assigned_path)
print(f'{guide.notna().sum():,} of {len(guide):,} cells kept')

1,608 of 1,716 cells kept


## 5. Columns, labels and gene names

Labels are still guide names. `cx.normalise_perturbation_labels` turns them into targets: `target_pattern` captures the gene from the library's naming, and control names (`NTC`, `non-targeting`, `scramble`, ...) become one `canonical_control`. `inplace=False` returns the result for a look before anything is written.

In [12]:
target = cx.normalise_perturbation_labels(
    assigned_path, column='guide', target_pattern=r'^(.+)_g\d+$', canonical_control='control', inplace=False,
)
obs_assigned = cx.load_obs(assigned_path)
obs_assigned['perturbation'] = target.astype('category')
cx.write_obs(assigned_path, obs_assigned)

truth = obs['perturbation'].astype(str).reindex(obs_assigned.index)
print(pd.crosstab(target.astype(str).rename('assigned'), truth.rename('label in the original file')))

label in the original file  AARS  AMIGO3  control
assigned                                         
AARS                         438       0        0
AMIGO3                         0     708        0
control                        0       0      462


Every assigned cell gets its original label; the simulated multiplets and empty captures are the cells left unassigned.

The analysis functions take the label column as `perturbation_column` (or `groupby`) and, when the gene symbols are not the `var` index, their column as `gene_name_column`. These helpers suggest which columns those are:

- `cx.infer_columns` returns both; `cx.detect_perturbation_column` and `cx.detect_gene_symbol_column` return one each. `None` for the gene column means the gene names are the `var` index, as here.
- `cx.ensure_gene_symbol_column` returns the gene symbols of an open file and complains when they look like Ensembl IDs.
- `cx.standardise_gene_names` strips Ensembl version suffixes and normalises the `mt-` prefix (here there is nothing to change).

In [13]:
print('infer_columns:', cx.infer_columns(assigned_path, verbose=False))
print('perturbation column:', cx.detect_perturbation_column(assigned_path, verbose=False))
print('gene symbol column:', cx.detect_gene_symbol_column(assigned_path, verbose=False))

backed = cx.read_backed(assigned_path)
print('symbols:', list(cx.ensure_gene_symbol_column(backed, None)[:3]))
backed.file.close()

cleaned = cx.standardise_gene_names(assigned_path, column='gene_ids', inplace=False)
print('gene IDs changed by standardise_gene_names:', int((cleaned.to_numpy() != cx.load_var(assigned_path)['gene_ids'].to_numpy()).sum()))

infer_columns: {'perturbation_column': 'perturbation', 'gene_name_column': None}
perturbation column: perturbation
gene symbol column: None
symbols: ['LINC00115', 'NOC2L', 'KLHL17']
gene IDs changed by standardise_gene_names: 0


## 6. Check the file

`cx.pp.check_data` reads `obs`, `var` and a few windows of the matrix and reports the mistakes that mislead an analysis without making it fail. Run on the guide column, it sees that the labels are not targets yet:

In [14]:
issues = cx.pp.check_data(assigned_path, perturbation_column='guide')

[cx] pp.check_data: error: Unable to infer control label automatically. Please provide 'control_label' explicitly.
[cx] pp.check_data: warning: 6 labels contain an underscore and are not measured genes (guide names?): 'NTC_g2', 'AMIGO3_g2', 'NTC_g1', 'AARS_g2', 'AARS_g1', ...
[cx] pp.check_data: warning: 3 var_names look like var_names_make_unique suffixes of other var_names, so one symbol sits on several rows: pp.collapse_genes(by=...) sums them; 'CRYBG3-1', 'IER3IP1-1', 'SRSF10-1'
[cx] pp.check_data: Done  1 errors, 2 warnings


On the target column one warning remains, about the genes. Each finding is a `cx.Issue` with a `level` and a `message`; `cx.raise_on_errors` stops a pipeline when any is an error.

In [15]:
issues = cx.pp.check_data(assigned_path, perturbation_column='perturbation')
cx.raise_on_errors(issues)      # no error-level issue: nothing is raised
[(issue.level, issue.message[:60] + '...') for issue in issues]

[cx] pp.check_data: warning: 3 var_names look like var_names_make_unique suffixes of other var_names, so one symbol sits on several rows: pp.collapse_genes(by=...) sums them; 'CRYBG3-1', 'IER3IP1-1', 'SRSF10-1'
[cx] pp.check_data: Done  0 errors, 1 warnings


[('warning',
  '3 var_names look like var_names_make_unique suffixes of othe...')]

## 7. One gene on several rows

The warning is real: this dataset carries three genes twice, the second row named `GENE-1` by `var_names_make_unique`. `cx.pp.collapse_genes` sums the rows that share a key you supply (a `var` column, or one key per gene). Here the key is the name without its suffix, for the names whose unsuffixed form is also a gene.

In [16]:
names = cx.load_var(assigned_path).index
base = names.str.replace(r'-\d+$', '', regex=True)
key = pd.Series(np.where(base.isin(names), base, names), index=names)
print('rows merged:', {name: key[name] for name in names[key.to_numpy() != names.to_numpy()]})

collapsed = cx.pp.collapse_genes(assigned_path, OUTPUT_DIR / 'screen_collapsed.h5ad', by=key)
print(collapsed.shape, '| genes made of two rows:', int((cx.load_var(collapsed.path)['n_source_genes'] > 1).sum()))

rows merged: {'CRYBG3-1': 'CRYBG3', 'IER3IP1-1': 'IER3IP1', 'SRSF10-1': 'SRSF10'}
[cx] pp.collapse_genes: Saving → tutorial_outputs/basics_data/screen_collapsed.h5ad
[cx] pp.collapse_genes: 11630 genes → 11627 (3 summed from several rows, 0 dropped for a missing key)
[cx] pp.collapse_genes: Done  1608 cells × 11627 genes, 5763493 non-zeros
(1608, 11627) | genes made of two rows: 3


## 8. Normalised values back to counts

Public data is often shipped as `log1p(normalize_total(counts))`, while count models need integers. `cx.pp.recover_counts` inverts the transform exactly and raises when the values are not consistent with one. To have such a file, the cell normalises the counts first (`cx.pp.normalize_total_log1p`, shown in the next notebook).

In [17]:
normalised = cx.pp.normalize_total_log1p(collapsed.path, OUTPUT_DIR / 'screen_log1p.h5ad', verbose=False)
recovered = cx.pp.recover_counts(normalised.path, OUTPUT_DIR / 'screen_recovered.h5ad')
print('recovered counts equal the originals:', (recovered.to_memory().X != collapsed.to_memory().X).nnz == 0)

[cx] pp.recover_counts: Saving → tutorial_outputs/basics_data/screen_recovered.h5ad
[cx] pp.recover_counts: Done  1608 cells × 11627 genes, 5763493 counts recovered
recovered counts equal the originals: True


## 9. Several files into one

A screen delivered as several files, often with the same control cells in each, is joined on disk by `cx.pp.concat_h5ad`: cells are matched by name, a cell present in several files is kept once, and `label` records each cell's source file. The cell splits the dataset into two overlapping files to show it. The result keeps the genes every file has; files that list only the genes detected in them (many sample-by-sample pipelines) take `join='outer'`, which keeps every gene as zeros where a file lacks it.

In [18]:
n = collapsed.shape[0]
every_gene = np.ones(collapsed.shape[1], dtype=bool)
parts = []
for name, cells in (('part_a', np.arange(n) < 1000), ('part_b', np.arange(n) >= 800)):     # 200 cells in both
    parts.append(OUTPUT_DIR / f'{name}.h5ad')
    cx.write_filtered_subset(collapsed.path, cell_mask=cells, gene_mask=every_gene, output_path=parts[-1])

joined = cx.pp.concat_h5ad(parts, output_path=OUTPUT_DIR / 'screen_joined.h5ad', label='source')
print(joined.shape, cx.load_obs(joined.path)['source'].value_counts().to_dict())

[cx] pp.concat_h5ad: 200 cells of tutorial_outputs/basics_data/part_b.h5ad already in an earlier file are dropped
[cx] pp.concat_h5ad: Done  1608 cells × 11627 genes
[cx] pp.concat_h5ad: Saving → tutorial_outputs/basics_data/screen_joined.h5ad
(1608, 11627) {'part_a': 1000, 'part_b': 608}


## 10. Layout, compression and disk space

- **Layout.** Functions that stream cells (QC, normalisation, NB-GLM) read a CSR file fast; those that stream genes (the Wilcoxon test) read CSC fast, and convert a copy on their own when it pays. `cx.pp.convert_to_csc` and `cx.pp.convert_to_csr` do it once, explicitly.
- **Compression.** `cx.compress_h5ad` re-encodes a file with gzip, losslessly; every HDF5 reader opens the result.
- **Disk.** `cx.tl.estimate_disk_usage` reports what a function will write before it runs. crispyx also warns from inside a call when space looks tight.

See [Working with large data](../large_data.rst) for when each matters.

In [19]:
print(cx.tl.estimate_disk_usage('convert_to_csc', collapsed.path))

csc = cx.pp.convert_to_csc(collapsed.path, output_path=OUTPUT_DIR / 'screen_csc.h5ad', verbose=False)
csr = cx.pp.convert_to_csr(csc.path, output_path=OUTPUT_DIR / 'screen_csr.h5ad', verbose=False)
print('CSC then CSR gives the same matrix:', (csr.to_memory().X != collapsed.to_memory().X).nnz == 0)

report = cx.compress_h5ad(csr.path, OUTPUT_DIR / 'screen_compressed.h5ad')
size = lambda path: Path(path).stat().st_size / 1e6
print(f"compressed: {size(csr.path):.1f} MB -> {size(OUTPUT_DIR / 'screen_compressed.h5ad'):.1f} MB")

{'output': DiskEstimate(required_bytes=93703454.0, free_bytes=3047811108864.0, path=PosixPath('tutorial_outputs/basics_data'))}
CSC then CSR gives the same matrix: True
compressed: 46.8 MB -> 8.3 MB


## Functions in this notebook

| Function | What it does | Where |
|---|---|---|
| `cx.read_h5ad_ondisk`, `cx.AnnData` | open an `.h5ad` file, matrix left on disk | section 1 |
| `cx.read_backed` | anndata's backed object, sparse layers left on disk | 1 |
| `cx.resolve_data_path` | the file behind a path, a `cx.AnnData` or a backed anndata object: what every function accepts as its data | 1 |
| `cx.load_obs`, `cx.load_var`, `cx.write_obs`, `cx.write_var` | read and write one metadata table without the matrix (`write_var` works as `write_obs` does) | 1, 4 |
| `cx.read_10x_mtx`, `cx.read_mtx` | a MatrixMarket folder or file into `.h5ad`, streamed | 2 |
| `cx.read_csv`, `cx.read_text` | a text table into `.h5ad`, streamed | 3 |
| `cx.write_filtered_subset` | any subset of cells and genes to a new file | 2, 4, 9 |
| `cx.pp.assign_guides` | the dominant guide per cell | 4 |
| `cx.normalise_perturbation_labels` | guide names to targets, control names to one label | 5 |
| `cx.infer_columns`, `cx.detect_perturbation_column`, `cx.detect_gene_symbol_column`, `cx.ensure_gene_symbol_column` | find the label and gene-symbol columns | 5 |
| `cx.standardise_gene_names` | strip Ensembl versions, normalise `mt-` | 5 |
| `cx.pp.check_data`, `cx.Issue`, `cx.raise_on_errors` | report mistakes in labels, genes and counts | 6 |
| `cx.pp.collapse_genes` | sum the rows of one gene | 7 |
| `cx.pp.recover_counts` | integer counts from log-normalised values | 8 |
| `cx.pp.concat_h5ad` | join files along cells, shared cells once | 9 |
| `cx.pp.convert_to_csc`, `cx.pp.convert_to_csr` | change the matrix layout | 10 |
| `cx.compress_h5ad` | lossless gzip re-encoding | 10 |
| `cx.tl.estimate_disk_usage` | disk a function will need | 10 |

Next: [QC, normalisation and embedding](basics_2_qc_embedding.ipynb). More on each step: [Loading and preparing data](../data_preparation.rst).